# Evaluate the revised MSR without the unused Transformer encoder

This notebook loads the existing Kermany test split, instantiates `MSR_revised`,
loads the original checkpoint while transferring the original decoder LayerNorm
weights to the revised `out_norm` layers, and evaluates the unchanged test set.

In [12]:
# Run from the project repository root.
from pathlib import Path
import sys
import copy
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "utils" / "config.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find utils/config.py in the current directory or its parents. "
        "Set PROJECT_ROOT to the project repository root."
    )
sys.path.insert(0, str(PROJECT_ROOT))

from utils.config import *
from data.dataset_loader import load_split_csvs, load_data
from model.MSR_revised import MSR, FocalLossCE
from utils.train import evaluate

# Explicitly lock the requested evaluation configuration.
assert DATASET == "kermany"
assert SPLIT_SEED == 42
assert not AUGMENT and not USE_MASK and not MASK_ONLY
assert not INPUT_MASKING and not FEATURE_MASKING and not USE_INVERTED_MASK
assert USE_ECA and not USE_DSA
assert not ENCODER_CONFIG

CSV_DIR = PROJECT_ROOT / "csv" / "kermany"
CHECKPOINT_PATH = PROJECT_ROOT / "model" / "weights" / "eca_LN_kermany42.pth"
assert CHECKPOINT_PATH.is_file(), f"Checkpoint not found: {CHECKPOINT_PATH}"

# Match the class count and architecture configuration used to train the checkpoint.
NUM_CLASSES = 2
model = MSR(
    layers=LAYERS,
    num_classes=NUM_CLASSES,
    use_eca=USE_ECA,
    use_dsa=USE_DSA,
).to(device)


In [15]:
def load_original_checkpoint_into_revised_model(model, checkpoint_path):
    """Transfer compatible weights; discard only the unused encoder parameters."""
    state = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    # Support checkpoints saved either as a raw state_dict or a wrapped dict.
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    # Remove common DataParallel prefix.
    state = {k.removeprefix("module."): v for k, v in state.items()}

    revised_state = model.state_dict()
    transferred = {}
    discarded_encoder = []
    ignored = []

    for key, value in state.items():
        if ".encoder." in key:
            discarded_encoder.append(key)
            continue
        if ".out_norm." in key:
            ignored.append((key, "<discarded unused original out_norm>", tuple(value.shape)))
            continue

        if ".decoder.norm." in key:
            new_key = key.replace(".decoder.norm.", ".out_norm.")
        else:
            new_key = key

        if new_key in revised_state and revised_state[new_key].shape == value.shape:
            transferred[new_key] = value
        else:
            ignored.append((key, new_key, tuple(value.shape)))

    missing = sorted(set(revised_state) - set(transferred))
    if missing:
        print("Missing revised keys:", missing)
        print("Ignored/incompatible original keys:", ignored[:20])
        raise RuntimeError("Checkpoint mapping is incomplete; inspect key mismatches before evaluation.")

    model.load_state_dict(transferred, strict=True)
    print(f"Loaded {len(transferred)} tensors.")
    print(f"Discarded {len(discarded_encoder)} unused encoder tensors.")
    print("\nIgnored or incompatible checkpoint keys:")
    for old_key, new_key, shape in ignored:
        print(f"  {old_key} -> {new_key} | shape={shape}")

    print("\nDiscarded encoder keys:")
    for key in discarded_encoder:
        print(f"  {key}")
    return model

model = load_original_checkpoint_into_revised_model(model, CHECKPOINT_PATH)
model.eval()


Loaded 326 tensors.
Discarded 24 unused encoder tensors.

Ignored or incompatible checkpoint keys:
  cross_scale_attention.in_proj_weight -> cross_scale_attention.in_proj_weight | shape=(768, 256)
  cross_scale_attention.in_proj_bias -> cross_scale_attention.in_proj_bias | shape=(768,)
  cross_scale_attention.out_proj.weight -> cross_scale_attention.out_proj.weight | shape=(256, 256)
  cross_scale_attention.out_proj.bias -> cross_scale_attention.out_proj.bias | shape=(256,)

Discarded encoder keys:
  transformer_cat.encoder.layers.0.self_attn.in_proj_weight
  transformer_cat.encoder.layers.0.self_attn.in_proj_bias
  transformer_cat.encoder.layers.0.self_attn.out_proj.weight
  transformer_cat.encoder.layers.0.self_attn.out_proj.bias
  transformer_cat.encoder.layers.0.linear1.weight
  transformer_cat.encoder.layers.0.linear1.bias
  transformer_cat.encoder.layers.0.linear2.weight
  transformer_cat.encoder.layers.0.linear2.bias
  transformer_cat.encoder.layers.0.norm1.weight
  transformer_

MSR(
  (backbone): Backbone(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d()
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d()
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d()
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d()
          (relu): ReLU(inplace=True)
          (downsample): Sequential(
            (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): FrozenBatchNorm2d()
          )
        )
        (1): Bottleneck(
          (conv1): Conv

In [16]:
# Load the exact saved train/val/test CSVs; use only the test loader for evaluation.
splits = load_split_csvs(
    csv_dir=CSV_DIR,
    dataset="kermany",
    seed=SPLIT_SEED,
    root=str(EXPECTED_DIR),
)
train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]

# load_data expects all three frames but constructs the test loader with val transforms
_, _, test_loader = load_data(train_df, val_df, test_df)

print("Test images:", len(test_df))
print("Class counts:")
print(test_df["class_name"].value_counts().sort_index())


2026-11-20 18:29:22 | Loaders: train 4684 | val 586 | test 586 | batch 128 | masks OFF
Test images: 586
Class counts:
class_name
NORMAL       158
PNEUMONIA    428
Name: count, dtype: int64


In [ ]:
criterion = FocalLossCE()
test_loss, test_accuracy, y_true, y_pred, y_prob = evaluate(
    model,
    test_loader,
    criterion,
    device,
    invert_mask=False,
    feature_masking=False,
    input_masking=False,
)

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

print(f"Test loss: {test_loss:.6f}")
print(f"Test accuracy: {test_accuracy:.4f}%")
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=sorted(test_df["class_name"].unique()), digits=4))

results = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "prob_class_0": y_prob[:, 0],
    "prob_class_1": y_prob[:, 1],
})
# results.to_csv("kermany_test_pred_eval.csv", index=False)


Test loss: 0.021698
Test accuracy: 98.1229%
Confusion matrix:
[[152   6]
 [  5 423]]
              precision    recall  f1-score   support

      NORMAL     0.9682    0.9620    0.9651       158
   PNEUMONIA     0.9860    0.9883    0.9872       428

    accuracy                         0.9812       586
   macro avg     0.9771    0.9752    0.9761       586
weighted avg     0.9812    0.9812    0.9812       586



In [23]:
from utils.complexity import *

compute_complexity(model)

2026-11-20 18:43:41 | Total parameters: 26,704,712 | trainable: 3,249,800
2026-11-20 18:43:41 | Inference time (1 image, CPU, single pass): 267.7 ms
Total:21.49 GMac
Module:  Global
aten.convolution: 21.49 GMac
aten.addmm: 164.48 KMac

Module:  backbone
aten.convolution: 19.08 GMac

Module:  conv_red_1
aten.convolution: 51.38 MMac

Module:  conv_red_2
aten.convolution: 102.76 MMac

Module:  conv_red_3
aten.convolution: 411.04 MMac

Module:  enhancement_cat
aten.convolution: 924.84 MMac

Module:  enhancement_3
aten.convolution: 924.84 MMac

Module:  classifier
aten.addmm: 164.48 KMac

2026-11-20 18:43:42 | MACs: 21,492,353,154


## Optional: evaluation-output equivalence check

Instantiate the **original** `MSR` from the original `model.MSR` and load the untouched checkpoint into it. 
Then compare logits on a fixed, non-shuffled batch against the revised model above.
The original model must use `ENCODER_CONFIG=False` and be in `eval()` mode.

In [ ]:
from model.MSR import MSR as OriginalMSR
from utils.load_model import *
original_model = OriginalMSR(
    layers=LAYERS, num_classes=NUM_CLASSES, use_eca=USE_ECA, use_dsa=USE_DSA
).to(device)
original_state = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if isinstance(original_state, dict) and "state_dict" in original_state:
    original_state = original_state["state_dict"]
original_state = {k.removeprefix("module."): v for k, v in original_state.items()}

original_model, device = load_model(CHECKPOINT_PATH, num_classes=2)
original_model.eval()

fixed_batch = next(iter(test_loader))
fixed_images, fixed_targets = fixed_batch[0].to(device), fixed_batch[-1].to(device)
with torch.inference_mode():
    old_logits = original_model(fixed_images)
    new_logits = model(fixed_images)
diff = (old_logits - new_logits).abs()
print("max_abs_diff:", diff.max().item())
print("mean_abs_diff:", diff.mean().item())
print("allclose:", torch.allclose(old_logits, new_logits, rtol=1e-5, atol=1e-6))



Current model parameters not loaded from checkpoint:
  transformer_3.out_norm.bias
  transformer_3.out_norm.weight
  transformer_cat.out_norm.bias
  transformer_cat.out_norm.weight

Custom model loaded successfully!
max_abs_diff: 0.0
mean_abs_diff: 0.0
allclose: True
